In [1]:
import os
import sys

import hydra
from dotenv import load_dotenv
from loguru import logger
from omegaconf import DictConfig, OmegaConf

from pathlib import Path
from kge.common.actions.kge_loader_action import KGELoader
from kge.common.inference_tasks import INFERENCE_TASK_MAP
from kge.utils import load_version

import torch
from datasets.common.constants import DataSplits

import os

/home/pablo.sanchez2/Documents/GitHub/project_spaice_ds/packages/pip/kge/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(override=False)

True

In [3]:
device = "cuda"
kge_version = "1.1.0"
package_dir = Path("/home/pablo.sanchez2/Documents/GitHub/project_spaice_ds/packages/pip/kge")
experiment_dir = package_dir /  f"outputs/prod/kge-v{kge_version}"

score_scaler_json_path = experiment_dir / "seed_0/model_checkpoint/last_score_scaler.json"

In [4]:

experiment_data = KGELoader.load_experiment(
        experiment_folder=experiment_dir,
        load_negative_sampler = True,
        device=device,
    )

2025-09-23 09:54:06.027 | INFO     | kge.common.actions.kge_train:prepare_data:147 - Data Repo: <datasets.data_repo.dsv.repo.DSVKGDataset object at 0x79945471ece0>
2025-09-23 09:54:06.759 | INFO     | datasets.data_repo.base:load_data:252 - Loaded knowledge graph from cache /home/pablo.sanchez2/Documents/GitHub/data/hakken_bio/pubtator3-v0.4.0/cached/v1.1.0
2025-09-23 09:54:06.760 | INFO     | kge.common.actions.kge_train:prepare_data:150 - Number of entities 570505
2025-09-23 09:54:06.760 | INFO     | kge.common.actions.kge_train:prepare_data:151 - Number of relations 12
2025-09-23 09:54:06.761 | INFO     | kge.common.actions.kge_train:prepare_data:153 - Number of training_facts 11236778
2025-09-23 09:54:06.763 | INFO     | kge.common.actions.kge_train:prepare_data:157 - Data Processor: <kge.data_processor.base.KGDataProcessor object at 0x799452b53a90>


In [5]:
model = experiment_data.model
data_processor = experiment_data.data_processor
negative_sampler = experiment_data.negative_sampler

success = model.load_score_scaler(score_scaler_json_path)
model.eval()

2025-09-23 09:54:19.371 | INFO     | kge.models.base:load_score_scaler:264 - Loading score scaler from /home/pablo.sanchez2/Documents/GitHub/project_spaice_ds/packages/pip/kge/outputs/prod/kge-v1.1.0/seed_0/model_checkpoint/last_score_scaler.json


ComplEx(
  (_entity_embeddings): Embedding(570505, 256)
  (_relation_embeddings): Embedding(12, 256)
)

In [6]:
dataloader = data_processor.get_data_loader(DataSplits.TRAIN)
if not success:
    print("Fitting")
    model.fit_score_scaler(data_loader=dataloader, json_path=score_scaler_json_path)

In [7]:

for batch in dataloader:
    sro_batch_pos: torch.Tensor = batch[0].to(device)
    scores_pos = model.score(sro_batch_pos)
    norm_scores_pos = model.normalize_scores(scores_pos)
    break


In [9]:
norm_scores_pos.median()

tensor(0.9909, device='cuda:0')

In [ ]:
model._scaler.config.target_eps

In [ ]:
f"{model._scaler.temperature.item():.2f}"

In [ ]:
import numpy as np
median = scores_pos.median().item()
eps = 1e-2
T = median/np.log((1 - eps) / eps)


def sigmoid(x: np.ndarray | float, b: float = 0.0, T: float = 1.0) -> np.ndarray | float:
    """
    Temperature-scaled sigmoid with shift b.

    f(x) = 1 / (1 + exp(-(x - b) / T))

    Args:
        x: Input scalar or NumPy array
        b: Baseline shift (default: 0.0)
        T: Temperature (steepness, > 0)

    Returns:
        Sigmoid-transformed values in (0, 1)
    """
    return 1.0 / (1.0 + np.exp(-(x - b) / T))    

In [ ]:
x = np.array([-100, 0.0, 4, 100, 500, 1000])

y = sigmoid(x, T=T)

np.round(y, 2) 
